# From Autograd to Architectures: Building, Training, and Debugging Neural Networks

**Author:** Plinio  |  **Stack:** Python, PyTorch, NumPy, scikit-learn, Matplotlib

This notebook is an end to end deep learning project in four parts. Each part builds a model from first principles, trains it, and (where it matters) walks through a real failure mode: **what broke, how it was diagnosed, and how it was fixed.**

| Part | Topic | What it demonstrates |
|---|---|---|
| 1 | Scalar autograd engine (micrograd style) | Backpropagation from scratch, topological sort, gradient checking against PyTorch |
| 2 | CNN (Convolutional Neural Network) on handwritten digits | Training loop, diagnosing a loss plateau, the "overfit one batch" test |
| 3 | Transformer built from scratch | Multi head self attention, a permutation equivariance bug, attention map analysis |
| 4 | GNN (Graph Neural Network): GCN (Graph Convolutional Network) | Message passing, adjacency normalization, graph vs no graph ablation |

Everything runs on a laptop CPU (Central Processing Unit) in a few minutes and needs no downloads: datasets are either bundled with scikit-learn or generated synthetically.

## 0. Setup and reproducibility

Deep learning results change from run to run because of random weight initialization and data shuffling. Fixing seeds for Python, NumPy, and PyTorch makes every number in this notebook reproducible. The device is set to a GPU (Graphics Processing Unit) through CUDA (Compute Unified Device Architecture) if one exists, otherwise the CPU.

In [ ]:
import math
import random
import time
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

---
# Part 1. A scalar autograd engine from scratch

Before using PyTorch, it is worth building the thing PyTorch does for us: **automatic differentiation**. The `Value` class below wraps one number. Every arithmetic operation creates a new `Value` that remembers its inputs (`_prev`) and a closure (`_backward`) that knows the *local* derivative of that operation.

Calling `.backward()` on the final output then does two things:

1. **Topological sort** of the computational graph, so every node is processed only after everything that depends on it.
2. **Chain rule** in reverse order: each node pushes `local_derivative * upstream_gradient` to its inputs. Gradients are *accumulated* (`+=`) because a value used twice receives gradient from both uses.

**Engineering detail.** The sort is an **iterative** depth first search with an explicit stack instead of recursion. Python's default recursion limit is 1000 frames, so a recursive sort crashes on any graph deeper than about 1000 operations (a long chain of additions, or an unrolled recurrent network). The iterative version has no such limit; this is tested a few cells below.

In [ ]:
class Value:
    # A scalar that records the operations applied to it, for reverse mode autodiff.

    def __init__(self, data: float, _children: tuple = (), _op: str = ""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    # ---- primitive operations: each defines its local gradient ----
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents are supported"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    # ---- reverse pass ----
    def backward(self) -> None:
        topo, visited = [], set()
        stack = [(self, False)]
        while stack:  # iterative post order DFS: children are appended before parents
            node, children_done = stack.pop()
            if children_done:
                topo.append(node)
                continue
            if node in visited:
                continue
            visited.add(node)
            stack.append((node, True))
            for child in node._prev:
                if child not in visited:
                    stack.append((child, False))
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    # ---- derived operations, built from the primitives above ----
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

### 1.1 A worked numerical example

Take the expression $L = \tanh(a \cdot b + c) \cdot f$ with $a=2,\ b=-0.5,\ c=0.3,\ f=-2$.

Forward pass: $a b = -1$, then $a b + c = -0.7$, then $\tanh(-0.7) = -0.6044$, and $L = -0.6044 \times -2 = 1.2087$.

Backward pass by hand:

* $\partial L / \partial f = \tanh(-0.7) = -0.6044$
* $\partial L / \partial (\text{tanh out}) = f = -2$
* $\partial \tanh / \partial z = 1 - \tanh^2 = 1 - 0.3653 = 0.6347$, so $\partial L / \partial z = -2 \times 0.6347 = -1.2695$
* $\partial L / \partial c = -1.2695$, $\ \partial L / \partial a = -1.2695 \times b = 0.6347$, $\ \partial L / \partial b = -1.2695 \times a = -2.5390$

The next cell checks the engine three ways: against these hand values, against **finite differences** $\frac{L(x+h) - L(x-h)}{2h}$, and against **PyTorch's autograd**.

In [ ]:
def build_expression(a, b, c, f):
    return (a * b + c).tanh() * f


inputs = {"a": 2.0, "b": -0.5, "c": 0.3, "f": -2.0}

# 1) micrograd
vals = {k: Value(v) for k, v in inputs.items()}
L = build_expression(**vals)
L.backward()
micro_grads = {k: v.grad for k, v in vals.items()}

# 2) central finite differences
def numerical_grad(name: str, h: float = 1e-6) -> float:
    plus, minus = dict(inputs), dict(inputs)
    plus[name] += h
    minus[name] -= h
    f = lambda d: math.tanh(d["a"] * d["b"] + d["c"]) * d["f"]
    return (f(plus) - f(minus)) / (2 * h)

# 3) PyTorch autograd (float64 for a tight comparison)
t = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in inputs.items()}
Lt = torch.tanh(t["a"] * t["b"] + t["c"]) * t["f"]
Lt.backward()

print(f"L = {L.data:.4f} (PyTorch: {Lt.item():.4f})\n")
print(f"{'input':>6} {'micrograd':>11} {'finite diff':>12} {'pytorch':>10}")
for k in inputs:
    fd, pt = numerical_grad(k), t[k].grad.item()
    print(f"{k:>6} {micro_grads[k]:>11.4f} {fd:>12.4f} {pt:>10.4f}")
    assert abs(micro_grads[k] - pt) < 1e-9 and abs(micro_grads[k] - fd) < 1e-5
print("\nAll gradient checks passed.")

### 1.2 Stress test: a graph 5000 operations deep

This is the recursion limit issue from above made concrete. We chain 5000 additions, $x_{i+1} = x_i + 0.001$, so the graph has depth 5000. The exact gradient of the last node with respect to the first is $1$ (addition passes gradient through unchanged). A recursive topological sort raises `RecursionError` here; the iterative one handles it.

In [ ]:
x0 = Value(1.0)
node = x0
for _ in range(5000):
    node = node + 0.001

t0 = time.perf_counter()
node.backward()
print(f"output = {node.data:.3f}, d(output)/d(x0) = {x0.grad}, "
      f"backward took {1000 * (time.perf_counter() - t0):.1f} ms")
assert x0.grad == 1.0

### 1.3 A neural network made only of `Value` objects

A **neuron** computes $\text{act}(w \cdot x + b)$. A **layer** is a list of neurons; an **MLP (Multi Layer Perceptron)** is a list of layers. Nothing here knows about "neural networks" as a special concept: it is just arithmetic on `Value`s, so backprop comes for free.

We train it on the two moons dataset (100 points, 2 features, labels $\pm 1$) with:

* **Hinge loss** $\max(0,\ 1 - y \cdot \hat{y})$: zero once a point is classified correctly with margin at least 1.
* **L2 (squared weight) regularization** with strength $\alpha = 10^{-4}$ to keep weights small.
* SGD (Stochastic Gradient Descent) with a learning rate decaying from 1.0 to 0.1.

Pure Python is slow (every multiply is a Python object), so this takes under a minute. That slowness is exactly why frameworks use tensors, which Part 2 switches to.

In [ ]:
class Module:
    def parameters(self) -> list:
        return []

    def zero_grad(self) -> None:
        for p in self.parameters():
            p.grad = 0.0


class Neuron(Module):
    def __init__(self, n_in: int, nonlinear: bool = True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlinear = nonlinear

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlinear else act

    def parameters(self):
        return self.w + [self.b]


class Layer(Module):
    def __init__(self, n_in: int, n_out: int, **kw):
        self.neurons = [Neuron(n_in, **kw) for _ in range(n_out)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MicroMLP(Module):
    def __init__(self, n_in: int, sizes: list):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1], nonlinear=i < len(sizes) - 1)
                       for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


set_seed(1337)
X_moons, y_moons = make_moons(n_samples=100, noise=0.1, random_state=1)
y_moons = y_moons * 2 - 1  # {0, 1} -> {-1, +1}

micro_model = MicroMLP(2, [16, 16, 1])
print(f"parameters: {len(micro_model.parameters())}")


def micro_loss(model):
    scores = [model([Value(a), Value(b)]) for a, b in X_moons]
    hinge = sum((1 + -yi * si).relu() for yi, si in zip(y_moons, scores)) * (1.0 / len(scores))
    l2 = 1e-4 * sum(p * p for p in model.parameters())
    acc = np.mean([(yi > 0) == (si.data > 0) for yi, si in zip(y_moons, scores)])
    return hinge + l2, acc


micro_hist = []
t0 = time.perf_counter()
for step in range(60):
    loss, acc = micro_loss(micro_model)
    micro_model.zero_grad()
    loss.backward()
    lr = 1.0 - 0.9 * step / 60
    for p in micro_model.parameters():
        p.data -= lr * p.grad
    micro_hist.append((loss.data, acc))
    if step % 10 == 0 or step == 59:
        print(f"step {step:2d} | loss {loss.data:.4f} | accuracy {acc:.0%}")
print(f"training time: {time.perf_counter() - t0:.1f} s")

### 1.4 Visualizing what the network learned

The left panel shows loss and accuracy per step. The right panel evaluates the trained network on a grid covering the input plane and colors each region by the predicted class. A nonlinear boundary curling between the two moons is evidence that the hidden ReLU (Rectified Linear Unit) layers are doing real work; a linear model could only draw a straight line.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

losses, accs = zip(*micro_hist)
ax1.plot(losses, label="loss")
ax1b = ax1.twinx()
ax1b.plot(accs, color="tab:orange", label="accuracy")
ax1.set(xlabel="step", ylabel="loss", title="micrograd training")
ax1b.set_ylabel("accuracy")
ax1.legend(loc="center right"); ax1b.legend(loc="lower right")

xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 45), np.linspace(-1.0, 1.5, 45))
grid_scores = np.array([micro_model([Value(a), Value(b)]).data
                        for a, b in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
ax2.contourf(xx, yy, grid_scores > 0, alpha=0.35, cmap="coolwarm")
ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm", edgecolor="k", s=25)
ax2.set(title="Decision boundary", xlabel="$x_1$", ylabel="$x_2$")
plt.tight_layout(); plt.show()

---
# Part 2. CNN on handwritten digits, and debugging a training run

Now we move to PyTorch tensors. The dataset is scikit-learn's **digits**: 1797 grayscale images of size 8 x 8, pixel values 0 to 16, 10 classes. It ships with scikit-learn, so nothing is downloaded.

Preprocessing:

* Scale pixels to $[0, 1]$ by dividing by 16. Unscaled inputs make early activations large and training less stable.
* Stratified 80/20 train/validation split, so each digit appears in the same proportion in both sets.
* Reshape to `(N, 1, 8, 8)`, the `(batch, channels, height, width)` layout convolution layers expect.

In [ ]:
digits = load_digits()
X_dig = (digits.images / 16.0).astype(np.float32)[:, None, :, :]
y_dig = digits.target.astype(np.int64)

X_tr, X_va, y_tr, y_va = train_test_split(X_dig, y_dig, test_size=0.2,
                                          stratify=y_dig, random_state=42)
dig_train = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                       batch_size=64, shuffle=True)
dig_val = DataLoader(TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va)),
                     batch_size=256)
print(f"train {X_tr.shape}, val {X_va.shape}")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(X_tr[np.where(y_tr == d)[0][0], 0], cmap="gray_r")
    ax.set_title(str(d)); ax.axis("off")
plt.suptitle("One training example per class", y=1.08); plt.show()

### 2.1 Reusable training utilities

Rather than rewriting a loop per model, one `train` function serves Parts 2 and 3. It records three signals every good debugging session needs:

* **Training loss** per epoch: is the model fitting at all?
* **Validation accuracy** per epoch: does fitting generalize?
* **Global gradient norm** per step, $\sqrt{\sum_i \lVert \nabla \theta_i \rVert^2}$: are gradients vanishing (near 0), exploding (very large), or healthy?

The loss call flattens logits to `(N, classes)` and targets to `(N,)`, which makes the same function work for image classification (one label per example) and sequence prediction (one label per token).

In [ ]:
@dataclass
class History:
    train_loss: list = field(default_factory=list)
    val_acc: list = field(default_factory=list)
    grad_norm: list = field(default_factory=list)


def global_grad_norm(model: nn.Module) -> float:
    norms = [p.grad.detach().norm(2) for p in model.parameters() if p.grad is not None]
    return torch.norm(torch.stack(norms), 2).item() if norms else 0.0


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        preds = model(xb).argmax(dim=-1)
        correct += (preds == yb).sum().item()
        total += yb.numel()
    return correct / total


def train(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
          epochs: int, lr: float, weight_decay: float = 0.0, verbose: bool = True) -> History:
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = History()
    for epoch in range(epochs):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            hist.grad_norm.append(global_grad_norm(model))
            opt.step()
            running += loss.item() * xb.size(0)
            seen += xb.size(0)
        hist.train_loss.append(running / seen)
        hist.val_acc.append(accuracy(model, val_loader))
        if verbose and (epoch % max(1, epochs // 5) == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:3d} | train loss {hist.train_loss[-1]:.4f} "
                  f"| val acc {hist.val_acc[-1]:.2%}")
    return hist

### 2.2 The CNN architecture

Three convolution blocks, each `Conv -> BatchNorm -> ReLU`:

* **3 x 3 convolutions with padding 1** keep the spatial size, so the 8 x 8 image stays 8 x 8 until pooling.
* **BatchNorm** normalizes each channel's activations per batch, which smooths optimization.
* **MaxPool 2 x 2** halves the resolution to 4 x 4 after two blocks, building some translation tolerance.
* **Global average pooling** collapses each of the 64 channels to one number, so the classifier head is only $64 \times 10 + 10 = 650$ weights instead of $64 \times 4 \times 4 \times 10$.

The constructor has an `apply_softmax` flag. It exists to reproduce a bug I hit on this exact model; see the next section.

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, n_classes: int = 10, apply_softmax: bool = False):
        super().__init__()
        def block(c_in, c_out):
            return [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU()]
        self.features = nn.Sequential(
            *block(1, 32), *block(32, 64),
            nn.MaxPool2d(2),                  # 8x8 -> 4x4
            *block(64, 64),
            nn.AdaptiveAvgPool2d(1),          # 4x4 -> 1x1
        )
        self.head = nn.Linear(64, n_classes)
        self.apply_softmax = apply_softmax

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.head(self.features(x).flatten(1))
        return logits.softmax(dim=-1) if self.apply_softmax else logits


n_params = sum(p.numel() for p in DigitCNN().parameters())
print(f"DigitCNN parameters: {n_params:,}")

### 2.3 First run: something is wrong

A common first version of a classifier ends the model with a softmax "so it outputs probabilities." The run below does exactly that. Watch the training loss.

In [ ]:
set_seed(0)
buggy_cnn = DigitCNN(apply_softmax=True)
hist_buggy = train(buggy_cnn, dig_train, dig_val, epochs=25, lr=3e-3)

### 2.4 Diagnosis: a loss floor that has nothing to do with the data

The training loss stalls well above zero even though a CNN should be able to memorize 1437 tiny images. When a model cannot fit its own training data, the first move is the **"overfit a single batch" test**: take one batch of 64 examples and train on it alone for a few hundred steps. Any correct model and loss pairing should drive loss to about 0. If it cannot, the bug is in the model or the loss, not in the data or hyperparameters.

**Why it fails here, with numbers.** `F.cross_entropy` expects raw logits and applies `log_softmax` internally. Passing it probabilities means softmax is applied **twice**. The inner softmax squashes every value into $[0, 1]$, so the "logits" the loss sees can differ by at most 1. The best possible case is the correct class at 1 and the other nine at 0:

$$\text{loss}_{\min} = -\log \frac{e^{1}}{e^{1} + 9 e^{0}} = \log\left(1 + \frac{9}{e}\right) = \log(4.311) \approx 1.461$$

So the loss can **never** go below 1.461, no matter how perfect the predictions. It also flattens gradients: the loss surface barely changes as the model grows more confident, so learning slows down.

In [ ]:
def overfit_one_batch(model: nn.Module, batch, steps: int = 300, lr: float = 3e-3) -> list:
    model.to(DEVICE).train()
    xb, yb = (t.to(DEVICE) for t in batch)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


theoretical_floor = math.log(1 + 9 / math.e)
one_batch = next(iter(dig_train))
set_seed(0); ob_buggy = overfit_one_batch(DigitCNN(apply_softmax=True), one_batch)
set_seed(0); ob_fixed = overfit_one_batch(DigitCNN(apply_softmax=False), one_batch)

print(f"theoretical floor with double softmax: {theoretical_floor:.4f}")
print(f"final single batch loss, buggy: {ob_buggy[-1]:.4f}")
print(f"final single batch loss, fixed: {ob_fixed[-1]:.6f}")

plt.figure(figsize=(7, 3.5))
plt.plot(ob_buggy, label="softmax + cross_entropy (bug)")
plt.plot(ob_fixed, label="raw logits + cross_entropy (fix)")
plt.axhline(theoretical_floor, ls="--", c="k", label=f"floor = {theoretical_floor:.3f}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Overfit one batch test"); plt.legend(); plt.show()

### 2.5 The fix and a full retrain

The fix is one line: return **raw logits** from `forward` and let the loss function handle the softmax. Probabilities, when needed for reporting, come from calling `softmax` at inference time outside the model. The same principle applies to `BCEWithLogitsLoss` versus `sigmoid` plus `BCELoss`; the fused versions are also more numerically stable because they use the log sum exp trick.

Small weight decay (L2 regularization through AdamW) is added for generalization now that the model can actually fit.

In [ ]:
set_seed(0)
fixed_cnn = DigitCNN(apply_softmax=False)
hist_fixed = train(fixed_cnn, dig_train, dig_val, epochs=25, lr=3e-3, weight_decay=1e-4)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(hist_buggy.train_loss, label="bug"); axes[0].plot(hist_fixed.train_loss, label="fix")
axes[0].axhline(theoretical_floor, ls="--", c="k", lw=1)
axes[0].set(title="Training loss", xlabel="epoch", yscale="log"); axes[0].legend()
axes[1].plot(hist_buggy.val_acc, label="bug"); axes[1].plot(hist_fixed.val_acc, label="fix")
axes[1].set(title="Validation accuracy", xlabel="epoch"); axes[1].legend()
axes[2].plot(hist_buggy.grad_norm, alpha=0.7, label="bug"); axes[2].plot(hist_fixed.grad_norm, alpha=0.7, label="fix")
axes[2].set(title="Global gradient norm", xlabel="step", yscale="log"); axes[2].legend()
plt.tight_layout(); plt.show()

### 2.6 Error analysis

Accuracy alone hides *which* mistakes the model makes. The confusion matrix shows true class (rows) against predicted class (columns); off diagonal cells are errors. Below it are the misclassified validation images, which usually turn out to be genuinely ambiguous strokes rather than random failures.

In [ ]:
fixed_cnn.eval()
with torch.no_grad():
    val_logits = fixed_cnn(torch.from_numpy(X_va).to(DEVICE)).cpu()
val_pred = val_logits.argmax(-1).numpy()
val_conf = val_logits.softmax(-1).max(-1).values.numpy()
cm = confusion_matrix(y_va, val_pred)
wrong = np.where(val_pred != y_va)[0]
print(f"final validation accuracy: {(val_pred == y_va).mean():.2%} ({len(wrong)} errors of {len(y_va)})")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set(xticks=range(10), yticks=range(10), xlabel="predicted", ylabel="true", title="Confusion matrix")
ax.grid(False); plt.colorbar(im); plt.show()

if len(wrong):
    n_show = min(len(wrong), 10)
    fig, axes = plt.subplots(1, n_show, figsize=(1.3 * n_show, 1.9), squeeze=False)
    for ax, idx in zip(axes[0], wrong[:n_show]):
        ax.imshow(X_va[idx, 0], cmap="gray_r"); ax.axis("off")
        ax.set_title(f"true {y_va[idx]}\npred {val_pred[idx]} ({val_conf[idx]:.0%})", fontsize=8)
    plt.suptitle("Misclassified validation images", y=1.15); plt.show()

---
# Part 3. A Transformer from scratch

The task: **reverse a sequence.** Input is 10 random digits (vocabulary of 10 tokens); the target at position $i$ is the input token at position $9 - i$.

| position | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| input | 3 | 7 | 1 | 1 | 9 | 0 | 4 | 2 | 8 | 5 |
| target | 5 | 8 | 2 | 4 | 0 | 9 | 1 | 1 | 7 | 3 |

It is a small task, but it has a useful property: it is **impossible without knowing token positions**, since the correct answer at position 0 depends on what sits at position 9. That makes it a clean test of whether attention and positional information work correctly.

With $10^{10}$ possible inputs, the 20,000 training sequences cover a vanishing fraction of the space, so validation accuracy measures a learned algorithm rather than memorization.

In [ ]:
SEQ_LEN, VOCAB = 10, 10


def make_reversal_data(n: int, seed: int) -> TensorDataset:
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(0, VOCAB, (n, SEQ_LEN), generator=g)
    return TensorDataset(x, x.flip(dims=[1]))


rev_train = DataLoader(make_reversal_data(20_000, seed=1), batch_size=128, shuffle=True)
rev_val = DataLoader(make_reversal_data(2_000, seed=2), batch_size=512)
xb, yb = next(iter(rev_val))
print("input :", xb[0].tolist())
print("target:", yb[0].tolist())

### 3.1 Multi head self attention

For each token, attention builds a **query** ("what am I looking for"), a **key** ("what do I contain"), and a **value** ("what I will pass on"). The output for token $i$ is a weighted average of all values, with weights from how well query $i$ matches every key:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

**Why divide by $\sqrt{d_k}$, numerically.** If query and key entries have variance 1, their dot product over $d_k = 16$ dimensions has variance 16, i.e. standard deviation 4. Scores like $\pm 8$ make softmax nearly one hot (for example $e^{8} / (e^{8} + e^{0}) = 0.9997$), and one hot softmax has almost zero gradient. Dividing by $\sqrt{16} = 4$ brings the standard deviation back to 1.

**Multiple heads** split $d_{model} = 64$ into 4 heads of 16 dimensions, so different heads can attend to different things. Q, K, and V are computed with one fused linear layer (one matrix multiply instead of three). The last attention map is cached in `self.last_attn` for visualization later.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.last_attn = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        # (B, T, C) -> (B, heads, T, d_head)
        q, k, v = (t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)   # (B, heads, T, T)
        attn = scores.softmax(dim=-1)
        self.last_attn = attn.detach()
        out = self.dropout(attn) @ v                                  # (B, heads, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


# shape and normalization sanity checks
mhsa = MultiHeadSelfAttention(d_model=64, n_heads=4)
out = mhsa(torch.randn(2, SEQ_LEN, 64))
assert out.shape == (2, SEQ_LEN, 64)
assert torch.allclose(mhsa.last_attn.sum(-1), torch.ones(2, 4, SEQ_LEN))
print("attention output:", tuple(out.shape), "| attention rows sum to 1: OK")

### 3.2 Transformer block and full model

Each block uses the **pre LayerNorm** arrangement used in GPT (Generative Pre-trained Transformer) style models:

$$x \leftarrow x + \text{Attention}(\text{LN}(x)), \qquad x \leftarrow x + \text{FFN}(\text{LN}(x))$$

The residual connections ($x + \dots$) give gradients a direct path back through the network, and normalizing *before* each sublayer tends to train more stably than normalizing after. The FFN (Feed Forward Network) expands to 128 dimensions with a GELU (Gaussian Error Linear Unit) activation and projects back to 64.

The model embeds tokens, optionally adds a learned **positional embedding**, runs 2 blocks, and projects each position to 10 logits. `use_pos_emb` is a flag because, as with the CNN, the first version of this model was missing something.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


class SeqTransformer(nn.Module):
    def __init__(self, vocab: int, seq_len: int, d_model: int = 64, n_heads: int = 4,
                 n_layers: int = 2, d_ff: int = 128, dropout: float = 0.0, use_pos_emb: bool = True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model) if use_pos_emb else None
        self.blocks = nn.ModuleList(TransformerBlock(d_model, n_heads, d_ff, dropout)
                                    for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(idx.size(1), device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


print(f"SeqTransformer parameters: {sum(p.numel() for p in SeqTransformer(VOCAB, SEQ_LEN).parameters()):,}")

### 3.3 First run: no positional embedding

The attention formula has no notion of order: it compares every token with every other token as a set. The first version relies on attention alone.

In [ ]:
set_seed(0)
tf_nopos = SeqTransformer(VOCAB, SEQ_LEN, use_pos_emb=False)
hist_nopos = train(tf_nopos, rev_train, rev_val, epochs=12, lr=2e-3)

### 3.4 Diagnosis: proving the model is blind to order

Token accuracy stalls far below 100%. The hypothesis: **without positional information, self attention is permutation equivariant.** Shuffling the input positions just shuffles the outputs the same way; nothing else changes. Formally, for any permutation $P$:

$$f(P x) = P f(x)$$

If that holds, then position 0 and position 9 holding the same token must produce the same prediction, so the model cannot tell "first" from "last" and reversal is impossible. The best it can do is guess the most common token among the other 9. A quick simulation over 50,000 random sequences puts that ceiling at **about 28.6% token accuracy**, so a run that plateaus around or below that number is consistent with the hypothesis.

A hypothesis like this should be **tested, not assumed.** The cell below permutes inputs and checks $f(Px) = P f(x)$ numerically for both versions of the model.

In [ ]:
def is_permutation_equivariant(model: nn.Module, x: torch.Tensor, seed: int = 0) -> bool:
    model.eval()
    perm = torch.randperm(x.size(1), generator=torch.Generator().manual_seed(seed))
    with torch.no_grad():
        permuted_output = model(x)[:, perm]   # P f(x)
        output_of_permuted = model(x[:, perm])  # f(P x)
    return torch.allclose(permuted_output, output_of_permuted, atol=1e-5)


x_test = next(iter(rev_val))[0][:32].to(DEVICE)
set_seed(0)
tf_pos_untrained = SeqTransformer(VOCAB, SEQ_LEN, use_pos_emb=True).to(DEVICE)
print("no positional embedding   -> equivariant:", is_permutation_equivariant(tf_nopos, x_test))
print("with positional embedding -> equivariant:", is_permutation_equivariant(tf_pos_untrained, x_test))

### 3.5 The fix: learned positional embeddings

Adding a learned vector per position ($10 \times 64 = 640$ extra parameters) to each token embedding breaks the symmetry: token "3" at position 0 now has a different representation from token "3" at position 9. The model can then learn "position $i$ should query for position $9 - i$."

Besides token accuracy, the cell reports **sequence accuracy**: the share of sequences reversed perfectly, all 10 tokens right. It is a much stricter metric: at 90% independent token accuracy, sequence accuracy would only be $0.9^{10} = 35\%$.

In [ ]:
set_seed(0)
tf_pos = SeqTransformer(VOCAB, SEQ_LEN, use_pos_emb=True)
hist_pos = train(tf_pos, rev_train, rev_val, epochs=12, lr=2e-3)


@torch.no_grad()
def sequence_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    hits = total = 0
    for xb, yb in loader:
        preds = model(xb.to(DEVICE)).argmax(-1).cpu()
        hits += (preds == yb).all(dim=1).sum().item()
        total += yb.size(0)
    return hits / total


for name, m in [("no pos emb", tf_nopos), ("with pos emb", tf_pos)]:
    print(f"{name:>13}: token acc {accuracy(m, rev_val):.2%} | sequence acc {sequence_accuracy(m, rev_val):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(hist_nopos.train_loss, label="no pos emb"); axes[0].plot(hist_pos.train_loss, label="with pos emb")
axes[0].set(title="Training loss", xlabel="epoch", yscale="log"); axes[0].legend()
axes[1].plot(hist_nopos.val_acc, label="no pos emb"); axes[1].plot(hist_pos.val_acc, label="with pos emb")
axes[1].axhline(0.1, ls="--", c="k", lw=1, label="chance")
axes[1].set(title="Validation token accuracy", xlabel="epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

### 3.6 Looking inside: attention maps

If the model truly learned reversal, some attention head should route information from position $9 - i$ to position $i$: an **anti diagonal** pattern in the attention matrix (rows are query positions, columns are key positions). The plot shows every head in every layer for one validation sequence. Checking internals like this confirms the model learned the intended mechanism rather than a shortcut.

In [ ]:
tf_pos.eval()
sample = next(iter(rev_val))[0][:1].to(DEVICE)
with torch.no_grad():
    pred = tf_pos(sample).argmax(-1)
print("input     :", sample[0].tolist())
print("prediction:", pred[0].tolist())

n_layers, n_heads = len(tf_pos.blocks), tf_pos.blocks[0].attn.n_heads
fig, axes = plt.subplots(n_layers, n_heads, figsize=(3 * n_heads, 3 * n_layers), squeeze=False)
for l, block in enumerate(tf_pos.blocks):
    for h in range(n_heads):
        ax = axes[l][h]
        ax.imshow(block.attn.last_attn[0, h].cpu(), cmap="viridis", vmin=0, vmax=1)
        ax.set_title(f"layer {l}, head {h}", fontsize=9)
        ax.set_xlabel("key position", fontsize=8); ax.set_ylabel("query position", fontsize=8)
        ax.grid(False)
plt.suptitle("Attention weights (look for the anti diagonal)", y=1.02); plt.tight_layout(); plt.show()

---
# Part 4. Graph Neural Networks: a GCN from scratch

Some data is naturally a **graph**: users and friendships, molecules and bonds, papers and citations. The task here is **node classification** with few labels, the setting of citation benchmarks like Cora, reproduced synthetically so it runs offline.

**Synthetic graph (Stochastic Block Model).** 400 nodes in 4 communities of 100. Two nodes in the same community are connected with probability 0.10; across communities, 0.01. The expected degree is about $99 \times 0.10 + 300 \times 0.01 \approx 13$, and roughly 77% of a node's neighbors share its label.

**Node features are deliberately weak.** Each node has a 16 dimensional feature vector: its community mean (entries drawn with standard deviation 0.25) plus Gaussian noise with standard deviation 1.0. Noise dominates signal, so features alone should classify poorly. The information that fixes this lives in the edges.

**Labels are scarce.** Only 20% of nodes (80) are labeled for training; 20% are for validation and 60% for testing.

In [ ]:
def make_sbm_graph(n_per_class=100, n_classes=4, p_in=0.10, p_out=0.01,
                   feat_dim=16, signal=0.25, noise=1.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    y = torch.arange(n_classes).repeat_interleave(n_per_class)
    n = y.numel()
    probs = torch.where(y[:, None] == y[None, :], p_in, p_out)
    upper = torch.triu((torch.rand(n, n, generator=g) < probs).float(), diagonal=1)
    A = upper + upper.T                                   # symmetric, no self loops
    means = torch.randn(n_classes, feat_dim, generator=g) * signal
    X = means[y] + torch.randn(n, feat_dim, generator=g) * noise
    perm = torch.randperm(n, generator=g)
    split = {"train": perm[: n // 5], "val": perm[n // 5: 2 * n // 5], "test": perm[2 * n // 5:]}
    return X, A, y, split


X_g, A_g, y_g, split = make_sbm_graph()
degrees = A_g.sum(1)
same_label = (A_g * (y_g[:, None] == y_g[None, :]).float()).sum() / A_g.sum()
print(f"nodes {A_g.size(0)}, edges {int(A_g.sum().item() // 2)}, "
      f"mean degree {degrees.mean():.1f}, same label neighbors {same_label:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(A_g, cmap="Greys", interpolation="nearest")
axes[0].set(title="Adjacency matrix (sorted by community)"); axes[0].grid(False)
axes[1].hist(degrees.numpy(), bins=20, edgecolor="k")
axes[1].set(title="Degree distribution", xlabel="degree", ylabel="nodes")
plt.tight_layout(); plt.show()

### 4.1 The GCN layer and why normalization matters

A GCN layer lets each node average its neighbors' features, then applies a shared linear map:

$$H' = \hat{A}\, H W, \qquad \hat{A} = D^{-1/2} (A + I) D^{-1/2}$$

* **$A + I$** adds self loops so each node keeps its own features in the average.
* **$D^{-1/2} \cdots D^{-1/2}$** (with $D$ the degree matrix of $A + I$) normalizes by degree.

**Why normalize, numerically.** Without it, aggregation is a *sum*. A node with 13 neighbors whose features are all about 1.0 gets a value of about 14 after one layer and about $14^2 = 196$ after two, while a node with 3 neighbors gets about $4^2 = 16$. Feature scale then tracks degree instead of content, and high degree nodes dominate the gradients. With symmetric normalization, the neighbor weights $1/\sqrt{d_i d_j}$ keep outputs on the input's scale. A useful consequence: the eigenvalues of $\hat{A}$ lie in $(-1, 1]$, so stacking layers does not blow up. The cell checks this and the symmetry numerically.

**A neat equivalence:** feed a GCN the identity matrix instead of $\hat{A}$ and it becomes an ordinary MLP, since each node then "aggregates" only itself. That gives a perfectly controlled ablation: **same code, same parameters, only the graph differs.**

In [ ]:
def normalize_adjacency(A: torch.Tensor) -> torch.Tensor:
    A_self = A + torch.eye(A.size(0))
    d_inv_sqrt = A_self.sum(1).pow(-0.5)
    return d_inv_sqrt[:, None] * A_self * d_inv_sqrt[None, :]


class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, A_norm: torch.Tensor) -> torch.Tensor:
        return A_norm @ self.linear(x)


class GCN(nn.Module):
    def __init__(self, in_dim: int, hidden: int, n_classes: int, dropout: float = 0.5):
        super().__init__()
        self.layer1, self.layer2 = GCNLayer(in_dim, hidden), GCNLayer(hidden, n_classes)
        self.dropout = dropout

    def embed(self, x, A_norm):
        x = F.dropout(x, self.dropout, self.training)
        return F.relu(self.layer1(x, A_norm))

    def forward(self, x, A_norm):
        h = F.dropout(self.embed(x, A_norm), self.dropout, self.training)
        return self.layer2(h, A_norm)


A_norm = normalize_adjacency(A_g)
eigs = torch.linalg.eigvalsh(A_norm)
assert torch.allclose(A_norm, A_norm.T)
print(f"A_hat symmetric: True | eigenvalue range [{eigs.min():.3f}, {eigs.max():.3f}]")

ones = torch.ones(A_g.size(0), 1)
A_raw = A_g + torch.eye(A_g.size(0))
hub, leaf = degrees.argmax().item(), degrees.argmin().item()
for name, M in [("sum (unnormalized)", A_raw), ("normalized", A_norm)]:
    two_hop = M @ (M @ ones)
    print(f"{name:>19}: after 2 layers, hub (deg {int(degrees[hub])}) = {two_hop[hub].item():7.2f}, "
          f"leaf (deg {int(degrees[leaf])}) = {two_hop[leaf].item():6.2f}")

### 4.2 Training: full batch with early stopping

The whole graph fits in memory, so every step uses all nodes (full batch), but the loss only counts the 80 **training** nodes. The other nodes still participate in message passing, which is what makes this semi supervised: unlabeled neighbors carry information to labeled ones.

With so few labels, overfitting is the main risk, so the setup follows the original GCN paper: dropout 0.5, weight decay $5 \times 10^{-4}$, and keeping the weights from the epoch with the **best validation accuracy**. Test accuracy is reported only for that checkpoint, so the test set never influences any choice.

In [ ]:
def train_node_classifier(model, X, A_used, y, split, epochs=300, lr=0.01, weight_decay=5e-4):
    X, A_used, y = X.to(DEVICE), A_used.to(DEVICE), y.to(DEVICE)
    split = {k: v.to(DEVICE) for k, v in split.items()}
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best = {"val": -1.0}
    curve = []
    for epoch in range(epochs):
        model.train()
        loss = F.cross_entropy(model(X, A_used)[split["train"]], y[split["train"]])
        opt.zero_grad(); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            pred = model(X, A_used).argmax(-1)
        acc = {k: (pred[idx] == y[idx]).float().mean().item() for k, idx in split.items()}
        curve.append((loss.item(), acc["train"], acc["val"]))
        if acc["val"] > best["val"]:
            best = {"val": acc["val"], "test": acc["test"], "epoch": epoch,
                    "state": {k: v.detach().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best["state"])
    return best, curve


results = {}
curves = {}
for name, A_used in [("MLP (identity, no edges)", torch.eye(A_g.size(0))), ("GCN (normalized graph)", A_norm)]:
    set_seed(0)
    model = GCN(in_dim=X_g.size(1), hidden=32, n_classes=4)
    best, curve = train_node_classifier(model, X_g, A_used, y_g, split)
    results[name], curves[name] = (model, best), curve
    print(f"{name:>26}: best val {best['val']:.2%} at epoch {best['epoch']:3d} -> test {best['test']:.2%}")

### 4.3 What the graph changes

Left: validation accuracy over training for both models. Right: the 32 dimensional hidden embeddings from the first layer, projected to 2D with PCA (Principal Component Analysis) and colored by true community. If message passing works, GCN embeddings should form tighter, better separated clusters than the MLP's, because each node's representation is an average over mostly same class neighbors, which cancels much of the feature noise. Averaging about 13 neighbors reduces the noise standard deviation by roughly $\sqrt{13} \approx 3.6$ times, and a second layer compounds it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for name, curve in curves.items():
    axes[0].plot([c[2] for c in curve], label=name)
axes[0].axhline(0.25, ls="--", c="k", lw=1, label="chance")
axes[0].set(title="Validation accuracy", xlabel="epoch"); axes[0].legend(fontsize=8)

for ax, (name, (model, best)) in zip(axes[1:], results.items()):
    A_used = torch.eye(A_g.size(0)) if name.startswith("MLP") else A_norm
    model.eval()
    with torch.no_grad():
        H = model.embed(X_g.to(DEVICE), A_used.to(DEVICE)).cpu().numpy()
    Z = PCA(n_components=2).fit_transform(H)
    ax.scatter(Z[:, 0], Z[:, 1], c=y_g.numpy(), cmap="tab10", s=10, alpha=0.8)
    ax.set(title=f"{name}\nhidden embeddings (PCA), test {best['test']:.1%}", xlabel="PC 1", ylabel="PC 2")
plt.tight_layout(); plt.show()

---
# Summary

The cell below collects the headline numbers from every experiment in one table.

In [ ]:
summary = [
    ("1. micrograd MLP", "two moons (train)", "none", f"{micro_hist[-1][1]:.1%}"),
    ("2. CNN", "digits (val)", "softmax before cross_entropy",
     f"{hist_buggy.val_acc[-1]:.1%} -> {hist_fixed.val_acc[-1]:.1%} (train loss {hist_buggy.train_loss[-1]:.3f} -> {hist_fixed.train_loss[-1]:.3f})"),
    ("3. Transformer", "reversal (val, token)", "no positional embedding", f"{hist_nopos.val_acc[-1]:.1%} -> {hist_pos.val_acc[-1]:.1%}"),
    ("4. GNN", "SBM graph (test)", "ignoring edges (MLP)",
     f"{results['MLP (identity, no edges)'][1]['test']:.1%} -> {results['GCN (normalized graph)'][1]['test']:.1%}"),
]
print(f"{'model':<18}{'dataset':<24}{'problem fixed':<32}{'metric (before -> after)'}")
print("-" * 120)
for row in summary:
    print(f"{row[0]:<18}{row[1]:<24}{row[2]:<32}{row[3]}")

### Lessons from the debugging

1. **Know the contract of your loss function.** `cross_entropy` wants logits. A double softmax does not crash; it quietly puts a floor of 1.461 under the loss. Computing that floor analytically turned a vague "training is slow" into a precise diagnosis.
2. **Overfit one batch first.** It separates model and loss bugs from data and hyperparameter problems in under a minute.
3. **Turn hypotheses into tests.** The positional embedding bug was confirmed by checking $f(Px) = P f(x)$ in code, not by guessing.
4. **Ablate to prove what matters.** Using the identity matrix as the adjacency turns the GCN into an MLP with the same code and parameters, so the accuracy gap is attributable to the graph alone.
5. **Verify internals, not only metrics.** Gradient checks against PyTorch and attention map inspection confirm the model works *for the right reason*.

### Possible extensions

* Replace the dense adjacency with a sparse tensor (`torch.sparse`) so the GCN scales to graphs with millions of nodes.
* Add causal masking to the Transformer and train it autoregressively as a character level language model.
* Swap the GCN for GAT (Graph Attention Network), connecting Parts 3 and 4: attention restricted to graph neighbors.
* Train the CNN on MNIST (Modified National Institute of Standards and Technology) or CIFAR-10 (Canadian Institute For Advanced Research) with data augmentation and a learning rate schedule.